# TailorTalk — AI-Powered Visual Saree Search

TailorTalk is an AI-powered visual search assistant for a saree catalogue.

The system allows a user to upload a saree image and retrieve visually similar
sarees from the catalogue using image embeddings and FAISS. A Gemini-powered
agent decides when the visual search tool should be used, while a Gradio
interface provides the user-facing application.

### Pipeline

User Image → Image Embedding → FAISS Similarity Search → Similar Sarees  
                                             ↓  
                                     Gemini Agent → Natural-language Response

In [46]:
import pandas as pd

df = pd.read_csv("data/saree.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())

Rows: 1074
Columns: 7


,Name,SKU,Stock,Retail Price,Discounted Price,image_url,Website Link
0,Pashmina - Banarasi Saree - Pink Colour QS204820,QS204820,1,6000,3150,https://byrappasilk.in/storage/uploads/bsrKlEU...,https://byrappasilks.in/shop/pashmina_banarasi...
1,Organza Tissue Sarees - White & Gold Colour QA...,QA255622,0,5495,5020,https://byrappasilk.in/storage/uploads/1cssgxd...,https://byrappasilks.in/shop/organza_tissue_sa...
2,Floral Organza Saree - Red Colour QA254685,QA254685,0,10995,10045,https://byrappasilk.in/storage/uploads/qg47mgX...,https://byrappasilks.in/shop/floral_organza_sa...
3,Munga Crape Sarees - Blue Colour AA313403,AA313403,0,6000,3150,https://byrappasilk.in/storage/uploads/DOh6Yh1...,https://byrappasilks.in/shop/munga_crape_saree...
4,Munga Crap saree With Black Colour AA313402,AA313402,3,6000,3150,https://byrappasilk.in/storage/uploads/4JLJoaY...,https://byrappasilks.in/shop/munga_crap_saree_...


## 1. Dataset Understanding

The first step is to inspect the catalogue structure, available fields,
missing values, and product metadata before building the visual search system.

In [47]:
print("Column names:")
print(df.columns.tolist())

print("\nMissing values:")
display(df.isnull().sum())

Column names:
['Name', 'SKU', 'Stock', 'Retail Price', 'Discounted Price', 'image_url', 'Website Link']

Missing values:


Name                0
SKU                 0
Stock               0
Retail Price        0
Discounted Price    0
image_url           0
Website Link        0
dtype: int64

### Duplicate and Uniqueness Checks

The catalogue is checked for duplicate SKUs and duplicate image URLs.

Duplicate SKUs are important for visual search because multiple catalogue
images can represent the same product. The final recommendation system will
therefore avoid returning the same SKU multiple times.

In [48]:
print("Duplicate SKUs:", df["SKU"].duplicated().sum())
print("Duplicate image URLs:", df["image_url"].duplicated().sum())

print("\nUnique SKUs:", df["SKU"].nunique())
print("Unique image URLs:", df["image_url"].nunique())

Duplicate SKUs: 419
Duplicate image URLs: 0

Unique SKUs: 655
Unique image URLs: 1074


In [49]:
duplicate_sku_rows = df[df["SKU"].duplicated(keep=False)].sort_values("SKU")

print("Number of rows belonging to duplicated SKUs:", len(duplicate_sku_rows))

display(
    duplicate_sku_rows[
        ["Name", "SKU", "Stock", "Retail Price", "Discounted Price", "image_url"]
    ].head(20)
)

Number of rows belonging to duplicated SKUs: 577


,Name,SKU,Stock,Retail Price,Discounted Price,image_url
929,Georgette Saree Lavender With Gandaberunda Mot...,AA200321,1,4900,2573,https://byrappasilk.in/storage/products/featur...
921,Georgette Saree Pink With Gandaberunda Motif A...,AA200321,2,4900,2573,https://byrappasilk.in/storage/products/featur...
922,Georgette Saree Yellow With Gandaberunda Motif...,AA200321,0,4900,-18374,https://byrappasilk.in/storage/products/featur...
923,Georgette Saree Peach With Gandaberunda Motif ...,AA200321,0,4900,2573,https://byrappasilk.in/storage/products/featur...
920,Georgette Saree Orange With Gandaberunda Motif...,AA200321,1,4900,2573,https://byrappasilk.in/storage/products/featur...
925,Georgette Saree Royal Purple With Gandaberunda...,AA200321,1,4900,2573,https://byrappasilk.in/storage/products/featur...
924,Georgette Saree Magenta With Gandaberunda Moti...,AA200321,1,4900,2573,https://byrappasilk.in/storage/products/featur...
927,Georgette Saree Navy Blue With Gandaberunda Mo...,AA200321,2,4900,2573,https://byrappasilk.in/storage/products/featur...
926,Georgette Saree Turquoise Blue With Gandaberun...,AA200321,0,4900,2573,https://byrappasilk.in/storage/products/featur...
928,Georgette Saree Yellow With Gandaberunda Motif...,AA200321,2,4900,2573,https://byrappasilk.in/storage/products/featur...


In [50]:
print("========== PRICE AND STOCK CHECK ==========")

print("Negative retail prices:", (df["Retail Price"] < 0).sum())
print("Negative discounted prices:", (df["Discounted Price"] < 0).sum())
print("Negative stock values:", (df["Stock"] < 0).sum())

print("\nDiscounted price greater than retail price:",
      (df["Discounted Price"] > df["Retail Price"]).sum())

print("\nMinimum retail price:", df["Retail Price"].min())
print("Minimum discounted price:", df["Discounted Price"].min())
print("Minimum stock:", df["Stock"].min())

========== PRICE AND STOCK CHECK ==========
Negative retail prices: 0
Negative discounted prices: 1
Negative stock values: 59

Discounted price greater than retail price: 0

Minimum retail price: 895
Minimum discounted price: -18374
Minimum stock: -6


## 2. Image Availability Check

The catalogue contains remote image URLs. Before downloading the dataset,
the URLs are checked so that unavailable images can be excluded from the
visual-search pipeline.

In [51]:
import requests
from concurrent.futures import ThreadPoolExecutor


def check_image_url(url):
    try:
        response = requests.get(
            url,
            timeout=10,
            stream=True
        )

        return response.status_code == 200

    except Exception:
        return False


urls = df["image_url"].tolist()

print("Checking", len(urls), "image URLs...")

with ThreadPoolExecutor(max_workers=10) as executor:
    results = list(executor.map(check_image_url, urls))


successful = sum(results)
failed = len(results) - successful

print("\n========== IMAGE URL RESULTS ==========")
print("Total URLs:", len(urls))
print("Successful:", successful)
print("Failed:", failed)

Checking 1074 image URLs...


KeyboardInterrupt: 

In [ ]:
failed_urls = []

for i, success in enumerate(results):
    if not success:
        failed_urls.append(i)

print("Failed row indices:", failed_urls)

print("\nFailed records:")

display(
    df.loc[
        failed_urls,
        ["Name", "SKU", "image_url", "Website Link"]
    ]
)

Failed row indices: [186, 289, 290, 291, 786]

Failed records:


,Name,SKU,image_url,Website Link
186,Tussar Saree With Madhubani Print Dusty Purple...,QS264566,https://byrappasilk.in/storage/uploads/fV3OCBj...,https://byrappasilks.in/shop/tussar_saree_with...
289,Royal Blue Pure Mysore Silk Saree with Golden ...,QS282741,https://byrappasilk.in/storage/uploads/qORoCvF...,https://byrappasilks.in/shop/royal_blue_pure_m...
290,Pure Mysore Silk Saree with pink & Contrast Bl...,QS270932,https://byrappasilk.in/storage/uploads/uG18LSU...,https://byrappasilks.in/shop/pure_mysore_silk_...
291,Tissue Saree With Lotus Printed QS282590,QS282590,https://byrappasilk.in/storage/uploads/rqAGmUv...,https://byrappasilks.in/shop/tissue_saree_with...
786,Pure Tussar Saree Kutch Work Beige AA204877,AA204877,https://byrappasilk.in/storage/products/featur...,https://byrappasilks.in/shop/pure_tussar_saree...


## 3. Downloading the Image Dataset

Accessible catalogue images are downloaded locally and stored using the
catalogue row index as the filename. This creates a stable mapping between
each image and its corresponding product metadata.

In [ ]:
import os
import requests
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor


# Create image folder
image_folder = "data/images"
os.makedirs(image_folder, exist_ok=True)


def download_image(row):
    index = row.name
    url = row["image_url"]

    try:
        response = requests.get(url, timeout=15)

        if response.status_code != 200:
            return index, False

        image = Image.open(BytesIO(response.content))
        image = image.convert("RGB")

        file_path = os.path.join(
            image_folder,
            f"{index}.jpg"
        )

        image.save(file_path, "JPEG")

        return index, True

    except Exception:
        return index, False


print("Downloading accessible saree images...")

with ThreadPoolExecutor(max_workers=10) as executor:
    download_results = list(
        executor.map(download_image, [row for _, row in df.iterrows()])
    )


successful_downloads = sum(
    success for _, success in download_results
)

failed_downloads = len(download_results) - successful_downloads


print("\n========== DOWNLOAD RESULTS ==========")
print("Total catalogue rows:", len(df))
print("Images downloaded:", successful_downloads)
print("Images failed:", failed_downloads)


========== DOWNLOAD RESULTS ==========
Total catalogue rows: 1074
Images downloaded: 1069
Images failed: 5


In [ ]:
import os

image_files = [
    f for f in os.listdir("data/images")
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
]

print("Images currently in folder:", len(image_files))
print("\nFirst 20 files:")
print(image_files[:20])

Images currently in folder: 1069

First 20 files:
['0.jpg', '1.jpg', '10.jpg', '100.jpg', '1000.jpg', '1001.jpg', '1002.jpg', '1003.jpg', '1004.jpg', '1005.jpg', '1006.jpg', '1007.jpg', '1008.jpg', '1009.jpg', '101.jpg', '1010.jpg', '1011.jpg', '1012.jpg', '1013.jpg', '1014.jpg']


### Download Verification

The downloaded image count is checked before generating embeddings.

## 4. Environment and Model Dependencies

The project uses PyTorch and Torchvision for image feature extraction and
Transformers-related dependencies required by the project environment.

In [ ]:
import torch
import torchvision
import transformers
from PIL import Image

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Transformers:", transformers.__version__)
print("All imports successful!")


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.5/122.1 MB 1.3 MB/s eta 0:01:35
   ---------------------------------------- 0.5/122.1 MB 1.3 MB/s eta 0:01:35
   ---------------------------------------- 0.8/122.1 MB 1.1 MB/s eta 0:01:51
   ---------------------------------------- 0.8/122.1 MB 1.1 MB/s eta 0:01:51
   ---------------------------------------- 1.0/122.1 MB 836.8 kB/s eta 0:02:25
   ---------------------------------------- 1.3/122.1 MB 845.5 kB/s eta 0:02:23
   ----------------------------

## 5. Visual Feature Extraction

A pretrained ResNet-50 model is used as a feature extractor rather than as
a classifier. The final classification layer is removed so that each saree
image is represented by a visual feature vector.

To improve fine-grained retrieval, two representations are generated:

1. Original colour image — retains colour and overall appearance.
2. Grayscale image — reduces dependence on colour and emphasizes structural
   visual information such as patterns, borders and design.

The two representations are combined before indexing.

In [57]:
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

# Load pretrained ResNet-50
weights = ResNet50_Weights.DEFAULT

resnet = resnet50(weights=weights)

# Remove classification layer
resnet.fc = nn.Identity()

resnet.eval()

# IMPORTANT:
# Use the SAME preprocessing for both catalogue images
# and user query images.
image_transform = weights.transforms()

print("ResNet-50 loaded successfully!")
print("Feature extractor ready.")
print("Using the pretrained model's official preprocessing.")

ResNet-50 loaded successfully!
Feature extractor ready.
Using the pretrained model's official preprocessing.


In [ ]:
%pip install faiss-cpu

   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
    --------------------------------------- 0.3/16.3 MB ? eta -:--:--
    --------------------------------------- 0.3/16.3 MB ? eta -:--:--
    --------------------------------------- 0.3/16.3 MB ? eta -:--:--
    --------------------------------------- 0.3/16.3 MB ? eta -:--:--
    --------------------------------------- 0.3/16.3 MB ? eta -:--:--
    ----------------


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [71]:
import os
import numpy as np
from PIL import Image
import torch


# ------------------------------------------------
# IMAGE DATASET
# ------------------------------------------------

image_folder = "data/images"

image_files = [
    f
    for f in os.listdir(image_folder)
    if f.lower().endswith(
        (".jpg", ".jpeg", ".png", ".webp")
    )
]

image_files.sort(
    key=lambda x: int(
        os.path.splitext(x)[0]
    )
)

print("Images found:", len(image_files))


# ------------------------------------------------
# GENERATE COLOUR + GRAYSCALE EMBEDDINGS
# ------------------------------------------------

colour_embeddings = []
grayscale_embeddings = []
valid_image_files = []


with torch.no_grad():

    for i, filename in enumerate(image_files):

        image_path = os.path.join(
            image_folder,
            filename
        )

        try:

            # Load image
            image = Image.open(
                image_path
            ).convert("RGB")


            # ------------------------------------------------
            # 1. COLOUR EMBEDDING
            # ------------------------------------------------

            colour_tensor = image_transform(
                image
            ).unsqueeze(0)

            colour_embedding = resnet(
                colour_tensor
            )

            colour_embedding = (
                colour_embedding
                .flatten(start_dim=1)
                .cpu()
                .numpy()
                .astype("float32")
            )


            # ------------------------------------------------
            # 2. GRAYSCALE / STRUCTURE EMBEDDING
            # ------------------------------------------------

            grayscale_image = (
                image
                .convert("L")
                .convert("RGB")
            )

            grayscale_tensor = image_transform(
                grayscale_image
            ).unsqueeze(0)

            grayscale_embedding = resnet(
                grayscale_tensor
            )

            grayscale_embedding = (
                grayscale_embedding
                .flatten(start_dim=1)
                .cpu()
                .numpy()
                .astype("float32")
            )


            # ------------------------------------------------
            # STORE EMBEDDINGS
            # ------------------------------------------------

            colour_embeddings.append(
                colour_embedding[0]
            )

            grayscale_embeddings.append(
                grayscale_embedding[0]
            )

            valid_image_files.append(
                filename
            )


        except Exception as e:

            print(
                f"Failed: {filename} -> {e}"
            )


        # Progress update
        if (i + 1) % 100 == 0:

            print(
                f"Processed "
                f"{i + 1}/{len(image_files)} images"
            )


# ------------------------------------------------
# CONVERT TO NUMPY ARRAYS
# ------------------------------------------------

colour_embeddings = np.array(
    colour_embeddings,
    dtype="float32"
)

grayscale_embeddings = np.array(
    grayscale_embeddings,
    dtype="float32"
)


# ------------------------------------------------
# RESULTS
# ------------------------------------------------

print("\n===== EMBEDDING RESULTS =====")

print(
    "Images embedded:",
    len(valid_image_files)
)

print(
    "Colour embedding shape:",
    colour_embeddings.shape
)

print(
    "Grayscale embedding shape:",
    grayscale_embeddings.shape
)

assert len(valid_image_files) == len(colour_embeddings) == len(grayscale_embeddings), (
    "Embedding arrays and image metadata are misaligned!"
)

print("Embedding arrays and metadata are aligned!")

Images found: 1069
Processed 100/1069 images
Processed 200/1069 images
Processed 300/1069 images
Processed 400/1069 images
Processed 500/1069 images
Processed 600/1069 images
Processed 700/1069 images
Processed 800/1069 images
Processed 900/1069 images
Processed 1000/1069 images

===== EMBEDDING RESULTS =====
Images embedded: 1069
Colour embedding shape: (1069, 2048)
Grayscale embedding shape: (1069, 2048)
Embedding arrays and metadata are aligned!


### Saving Visual Embeddings

The generated colour and grayscale embeddings are saved along with the
image-to-embedding mapping so that the FAISS index and catalogue metadata
remain aligned.

In [62]:
# Save colour embeddings
np.save(
    "data/colour_embeddings.npy",
    colour_embeddings
)

# Save grayscale embeddings
np.save(
    "data/grayscale_embeddings.npy",
    grayscale_embeddings
)

# Save image-to-embedding mapping
embedding_metadata = pd.DataFrame({
    "image_file": valid_image_files
})

embedding_metadata.to_csv(
    "data/embedding_metadata.csv",
    index=False
)

print("Embeddings saved successfully!")

print("Colour embeddings:")
print("data/colour_embeddings.npy")

print("Grayscale embeddings:")
print("data/grayscale_embeddings.npy")

print("Metadata:")
print("data/embedding_metadata.csv")

Embeddings saved successfully!
Colour embeddings:
data/colour_embeddings.npy
Grayscale embeddings:
data/grayscale_embeddings.npy
Metadata:
data/embedding_metadata.csv


## 6. Catalogue-to-Embedding Mapping

Each embedding is linked to its corresponding catalogue image and product
metadata. This mapping is required so that FAISS search results can be
translated back into product name, SKU, price, stock and website information.

In [ ]:
# Create image filename mapping to catalogue rows

saree_df["image_file"] = [
    f"{i}.jpg" for i in range(len(saree_df))
]

# Show the mapping
print(saree_df[[
    "image_file",
    "Name",
    "SKU",
    "Stock",
    "Retail Price",
    "Discounted Price"
]].head())

In [ ]:
# Keep only catalogue rows that have successfully downloaded images

search_catalogue = embedding_metadata.merge(
    saree_df,
    on="image_file",
    how="inner"
)

print("Searchable sarees:", len(search_catalogue))
print("\nColumns:")
print(search_catalogue.columns.tolist())

print("\nFirst 5 searchable sarees:")
print(
    search_catalogue[
        [
            "image_file",
            "Name",
            "SKU",
            "Stock",
            "Retail Price",
            "Discounted Price",
            "Website Link"
        ]
    ].head()
)

## 7. FAISS Vector Index

The combined visual embeddings are stored in a FAISS inner-product index.
After L2 normalization, inner product corresponds to cosine similarity.

The final representation gives more weight to structural information while
retaining colour information:

- Colour: 35%
- Structural/grayscale representation: 65%

This weighting is an experimental improvement intended to reduce purely
colour-driven matches while preserving useful colour similarity.

In [63]:
import faiss
import numpy as np

# Normalize individual embedding spaces
colour_embeddings = colour_embeddings.astype("float32")
grayscale_embeddings = grayscale_embeddings.astype("float32")

faiss.normalize_L2(colour_embeddings)
faiss.normalize_L2(grayscale_embeddings)


# Combine colour and structural information
colour_weight = 0.35
structure_weight = 0.65

combined_embeddings = (
    colour_weight * colour_embeddings
    + structure_weight * grayscale_embeddings
)

# Normalize combined vectors
faiss.normalize_L2(combined_embeddings)


# Create cosine-similarity FAISS index
index = faiss.IndexFlatIP(
    combined_embeddings.shape[1]
)

index.add(combined_embeddings)


print("FAISS index created successfully!")
print("Vectors:", index.ntotal)
print("Embedding shape:", combined_embeddings.shape)


# Save the CURRENT index
faiss.write_index(
    index,
    "data/saree_faiss.index"
)

print("FAISS index saved successfully!")

FAISS index created successfully!
Vectors: 1069
Embedding shape: (1069, 2048)
FAISS index saved successfully!


In [ ]:
%pip install langchain

   ---------------------------------------- 0.0/565.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/565.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/565.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/565.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/565.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/565.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/565.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/565.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/565.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/565.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/565.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/565.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/565.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/565.1 kB ? eta -:--:--
   ---


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 8. LangChain Visual Search Tool

The visual similarity search is exposed as a callable LangChain tool.

The tool accepts a query image path and the requested number of results.
It generates the same colour and structural embeddings used for the
catalogue, searches the FAISS vector index, and returns product metadata
along with similarity scores.

Multiple catalogue images can belong to the same SKU, so the tool removes
duplicate SKUs and returns unique products.

In [52]:
from langchain_core.tools import tool
import numpy as np
import pandas as pd
import faiss
import torch
from PIL import Image


# ------------------------------------------------
# LOAD CATALOGUE
# ------------------------------------------------

saree_df = pd.read_csv(
    "data/saree.csv"
)

# Create image filename mapping
saree_df["image_file"] = [
    f"{i}.jpg"
    for i in range(len(saree_df))
]


# ------------------------------------------------
# LOAD EMBEDDING METADATA
# ------------------------------------------------

embedding_metadata = pd.read_csv(
    "data/embedding_metadata.csv"
)


# ------------------------------------------------
# BUILD SEARCHABLE CATALOGUE
# ------------------------------------------------

search_catalogue = embedding_metadata.merge(
    saree_df,
    on="image_file",
    how="inner"
)

print(
    "Searchable sarees:",
    len(search_catalogue)
)


# ------------------------------------------------
# LANGCHAIN SEARCH TOOL
# ------------------------------------------------

@tool
def search_similar_sarees(
    image_path: str,
    top_k: int = 5
) -> list:
    """
    Find visually similar sarees using
    colour + structural image embeddings.
    """

    # ------------------------------------------------
    # LOAD QUERY IMAGE
    # ------------------------------------------------

    query_image = Image.open(
        image_path
    ).convert("RGB")


    # ------------------------------------------------
    # COLOUR EMBEDDING
    # ------------------------------------------------

    colour_tensor = image_transform(
        query_image
    ).unsqueeze(0)

    with torch.no_grad():
        colour_embedding = resnet(
            colour_tensor
        )

    colour_embedding = (
        colour_embedding
        .flatten(start_dim=1)
        .cpu()
        .numpy()
        .astype("float32")
    )


    # ------------------------------------------------
    # GRAYSCALE / STRUCTURE EMBEDDING
    # ------------------------------------------------

    grayscale_image = (
        query_image
        .convert("L")
        .convert("RGB")
    )

    grayscale_tensor = image_transform(
        grayscale_image
    ).unsqueeze(0)

    with torch.no_grad():
        grayscale_embedding = resnet(
            grayscale_tensor
        )

    grayscale_embedding = (
        grayscale_embedding
        .flatten(start_dim=1)
        .cpu()
        .numpy()
        .astype("float32")
    )


    # ------------------------------------------------
    # NORMALIZE EMBEDDINGS
    # ------------------------------------------------

    faiss.normalize_L2(
        colour_embedding
    )

    faiss.normalize_L2(
        grayscale_embedding
    )


    # ------------------------------------------------
    # COMBINE COLOUR + STRUCTURE
    # ------------------------------------------------

    query_embedding = (
        0.35 * colour_embedding
        + 0.65 * grayscale_embedding
    )


    # ------------------------------------------------
    # NORMALIZE FINAL EMBEDDING
    # ------------------------------------------------

    faiss.normalize_L2(
        query_embedding
    )


    # ------------------------------------------------
    # SEARCH MORE CANDIDATES
    # ------------------------------------------------
    # We search for more than top_k because
    # multiple images may belong to the same SKU.

    search_k = max(
        top_k * 5,
        25
    )

    scores, indices = index.search(
        query_embedding,
        search_k
    )


    # ------------------------------------------------
    # BUILD RESULTS
    # ------------------------------------------------

    results = []

    # Keep only one result per SKU
    seen_skus = set()


    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        row = search_catalogue.iloc[idx]

        sku = row["SKU"]


        # Skip duplicate products
        if sku in seen_skus:
            continue

        seen_skus.add(sku)


        results.append({
            "image_file": row["image_file"],
            "name": row["Name"],
            "sku": sku,
            "similarity_score": float(score),
            "retail_price": float(
                row["Retail Price"]
            ),
            "discounted_price": float(
                row["Discounted Price"]
            ),
            "stock": int(
                row["Stock"]
            ),
            "website_link": row["Website Link"]
        })


        # Stop once we have enough
        # UNIQUE products
        if len(results) >= top_k:
            break


    return results


print(
    "Improved TailorTalk search tool created!"
)

Searchable sarees: 1069
Improved TailorTalk search tool created!


In [70]:
import faiss
import pandas as pd
import numpy as np

# Load the saved FAISS index
index = faiss.read_index("data/saree_faiss.index")

# Load metadata
embedding_metadata = pd.read_csv(
    "data/embedding_metadata.csv"
)

print("FAISS index loaded!")
print("Vectors:", index.ntotal)
print("Metadata rows:", len(embedding_metadata))

assert index.ntotal == len(embedding_metadata), (
    "FAISS index and embedding metadata are misaligned!"
)

print("Index and metadata are aligned!")

FAISS index loaded!
Vectors: 1069
Metadata rows: 1069
Index and metadata are aligned!


### Visual Search Validation

The search tool is tested using an image that already exists in the catalogue.
An exact catalogue image is expected to retrieve itself with the highest
similarity score, providing a basic sanity check for the embedding and FAISS
pipeline.

In [66]:
# Test with an image that already exists in the catalogue

test_image = "data/images/0.jpg"

test_result = search_similar_sarees.invoke({
    "image_path": test_image,
    "top_k": 5
})

print("\n===== SEARCH RESULTS =====")

for i, result in enumerate(test_result, start=1):

    print(f"\n{i}. {result['name']}")
    print(f"Image: {result['image_file']}")
    print(f"SKU: {result['sku']}")
    print(f"Similarity: {result['similarity_score']:.4f}")
    print(f"Retail Price: ₹{result['retail_price']}")
    print(f"Discounted Price: ₹{result['discounted_price']}")
    print(f"Stock: {result['stock']}")
    print(f"Link: {result['website_link']}")


===== SEARCH RESULTS =====

1. Pashmina - Banarasi Saree - Pink Colour QS204820
Image: 0.jpg
SKU: QS204820
Similarity: 1.0000
Retail Price: ₹6000.0
Discounted Price: ₹3150.0
Stock: 1
Link: https://byrappasilks.in/shop/pashmina_banarasi_saree_pink_colour_qs204820_1747470887

2. Pashmina - Banarasi Saree -Navy Blue Colour QA255417
Image: 10.jpg
SKU: QA255417
Similarity: 0.8825
Retail Price: ₹5995.0
Discounted Price: ₹5480.0
Stock: 1
Link: https://byrappasilks.in/shop/pashmina_banarasi_saree_navy_blue_colour_qa255417_1747470807

3. Pashmina - Banarasi Saree -Cream Colour QA255621
Image: 9.jpg
SKU: QA255621
Similarity: 0.8605
Retail Price: ₹6000.0
Discounted Price: ₹3150.0
Stock: 1
Link: https://byrappasilks.in/shop/pashmina_banarasi_saree_cream_colour_qa255621_1750686637

4. Munga Mustard Yellow Banarasi Silk Saree with Intricate Prints  QS240177
Image: 119.jpg
SKU: QS240177
Similarity: 0.8185
Retail Price: ₹5995.0
Discounted Price: ₹5476.0
Stock: 0
Link: https://byrappasilks.in/shop/mun

In [67]:
%pip install -U langchain-google-genai

  Attempting uninstall: langchain-google-genai
    Found existing installation: langchain-google-genai 4.3.3
    Uninstalling langchain-google-genai-4.3.3:
      Successfully uninstalled langchain-google-genai-4.3.3
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 9. Gemini-Powered TailorTalk Agent

Gemini acts as the conversational layer of TailorTalk.

The agent understands whether the user is requesting a visual similarity
search. When visual search is required, it calls the LangChain
`search_similar_sarees` tool.

The agent returns catalogue information provided by the search tool
without inventing product details or modifying similarity scores.

In [53]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
import os

# Make sure your Gemini API key is available
# If you already set GEMINI_API_KEY earlier, this does nothing.
if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = input("Enter your Gemini API key: ")

# Create the Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# Create the TailorTalk agent
agent = create_agent(
    model=llm,
    tools=[search_similar_sarees],
    system_prompt="""
You are TailorTalk, an AI fashion assistant that helps users find visually
similar sarees from a saree catalogue.

Your main job is to understand what the user wants.

When the user asks to:
- find similar sarees
- search for sarees similar to an image
- recommend sarees based on an uploaded image
- find visually similar clothing

use the search_similar_sarees tool.

The tool requires:
- image_path: path to the query image
- top_k: number of results

When displaying search results, clearly show:
1. Saree name
2. Similarity score
3. Retail price
4. Discounted price
5. Stock
6. Product link

When showing similarity search results:
- Show only unique SKUs/products.
- Do not repeat the same SKU.
- Preserve the similarity scores returned by the tool.
- Do not invent or modify similarity scores.

Do not invent product information.

If the user is simply chatting or asking a general fashion question,
do not call the similarity-search tool.

Be concise, friendly, and helpful.
"""
)

print("TailorTalk agent created successfully!")

TailorTalk agent created successfully!


In [ ]:
# Test TailorTalk AI Agent

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Find me sarees similar to this image: data/images/0.jpg"
        }
    ]
})

print(response["messages"][-1].content)

Here are some sarees similar to the image you provided:

1.  **Pashmina - Banarasi Saree - Pink Colour QS204820**
    *   Similarity Score: 0.854
    *   Retail Price: ₹6000
    *   Discounted Price: ₹3150
    *   Stock: 1
    *   Product Link: https://byrappasilks.in/shop/pashmina_banarasi_saree_pink_colour_qs204820_1747470887

2.  **Pashmina - Banarasi Saree -Cream Colour QA255621**
    *   Similarity Score: 0.780
    *   Retail Price: ₹6000
    *   Discounted Price: ₹3150
    *   Stock: 1
    *   Product Link: https://byrappasilks.in/shop/pashmina_banarasi_saree_cream_colour_qa255621_1750686637

3.  **Pashmina - Banarasi Saree -Navy Blue Colour QA255417**
    *   Similarity Score: 0.773
    *   Retail Price: ₹5995
    *   Discounted Price: ₹5480
    *   Stock: 1
    *   Product Link: https://byrappasilks.in/shop/pashmina_banarasi_saree_navy_blue_colour_qa255417_1747470807

4.  **Munga Mustard Yellow Banarasi Silk Saree with Intricate Prints QS240177**
    *   Similarity Score: 0.749

In [ ]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Find me 3 sarees similar to this image: data/images/100.jpg"
        }
    ]
})

print(response["messages"][-1].content)

Here are 3 sarees similar to the image you provided:

1.  **Saree Name:** Silk Saree Peach With Golden Zari Border And Subtle Floral Embroidery Border QS236750
    **Similarity Score:** 0.938
    **Retail Price:** ₹7900
    **Discounted Price:** ₹4147
    **Stock:** 0
    **Product Link:** https://byrappasilks.in/shop/silk_saree_peach_with_golden_zari_border_and_subtle_floral_embroidery_border_qs236750_1754410501

2.  **Saree Name:** Silk Saree Yellow With Subtle Zari Work And Floral Motifs QS236745
    **Similarity Score:** 0.831
    **Retail Price:** ₹7900
    **Discounted Price:** ₹4148
    **Stock:** 1
    **Product Link:** https://byrappasilks.in/shop/silk_saree_yellow_with_subtle_zari_work_and_floral_motifs_qs236745_1754380527

3.  **Saree Name:** Organza Saree Embroidery Border Baby Pink QS315096
    **Similarity Score:** 0.794
    **Retail Price:** ₹3900
    **Discounted Price:** ₹2048
    **Stock:** 1
    **Product Link:** https://byrappasilks.in/shop/organza_saree_embroidery_

In [ ]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What saree would be good for a wedding?"
        }
    ]
})

print(response["messages"][-1].content)

[{'type': 'text', 'text': "I can help you find visually similar sarees if you have an image of a saree you like! Just upload the image, and I'll find similar ones for you.\n\nFor wedding sarees, people often look for rich fabrics like silk, georgette, or chiffon, with intricate embroidery, Zari work, or embellishments. Popular colors include red, maroon, gold, and pastels.", 'extras': {'signature': 'CrQDARFNMg/pi5Gw8hdMN4SiPQB+7k5Yx7svQ+ne4o1hG0R0YRZl8tfLKGnTnJ3jFyDGweJC0Nk2nW7w672zd4n0kniRoQX1FQD3BXQMXd5ZxI+aDo8StuFO08CNA6FThA0+jDXCkUkxEBsRNq2d1BrP4KK1iE+3tZVnUiU7RKgjEsN6CKfJtW56HKpNWFwP01QP1H/r8QUO/DKJ72s6OE6Ed9jw1rSX04zkt2PrGN23DZsH5HkSlOU0KY8fiYgpHUDfr3ob/uSjbK6M5W7vv8FAOIIkFPb6R1DJbBO2ZSaEsUjbLyGEVrTGoCnfFNooZVZW9jMtzLA37uvEGbAA2lhKnUkXWJRsb27z42W7TJBTLa+7SK+ixDmo0vY5eSL4ApxxnMP/wJWMgaX4dMbxp20DpLGBxYiAoKi8xZbJmRNDSURn14gh7yYIKhaoODZFkMHZMvZNAZArD6/4/5yZZ7qcNnkeT59zf9kIEJoYpYaAFkHxlQFBzhKFg6yPqvmE2YtVCPTccmUUM8jBbIMyUuc1z88hl9wFBbGRkwcDVl1QvaVJVb6yRPkddyIhIBxNyxnGz8gpsg=='}}]


In [68]:
%pip install -U gradio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 10. Gradio Frontend

The Gradio interface provides the user-facing TailorTalk application.

Users can upload a saree image and optionally describe what they are looking
for. The request is passed to the Gemini agent, which decides whether to call
the visual similarity search tool.

In [69]:
import gradio as gr
import os


def extract_agent_text(response):
    """
    Extract readable text from the Gemini/LangChain agent response.
    """

    try:
        content = response["messages"][-1].content

        # Gemini may return a list of content blocks
        if isinstance(content, list):

            text_parts = []

            for block in content:

                if isinstance(block, dict):
                    if block.get("type") == "text":
                        text_parts.append(block.get("text", ""))

                elif isinstance(block, str):
                    text_parts.append(block)

            return "\n".join(text_parts)

        return str(content)

    except Exception as e:
        return f"Could not read agent response: {e}"


def tailor_talk_search(image_path, user_message):
    """
    Connect the Gradio interface to the TailorTalk agent.
    """

    # Make sure an image was uploaded
    if not image_path:
        return "Please upload a saree image first."

    # Default message if the user leaves the text box empty
    if not user_message.strip():
        user_message = "Find sarees visually similar to this image."

    # Tell the agent where the uploaded image is located
    prompt = f"""
The user uploaded a saree image.

Image path:
{image_path}

User request:
{user_message}

Use the search_similar_sarees tool when the request requires
visual similarity search. Use the uploaded image path as the
image_path argument.

Return the most useful results clearly.
"""

    try:

        response = agent.invoke({
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        })

        return extract_agent_text(response)

    except Exception as e:

        return f"Error while running TailorTalk:\n\n{str(e)}"


# -----------------------------
# Gradio Interface
# -----------------------------

with gr.Blocks(title="TailorTalk - AI Saree Search") as demo:

    gr.Markdown(
        """
        # 👗 TailorTalk
        ### AI-Powered Visual Saree Search

        Upload a saree image and ask TailorTalk to find visually
        similar sarees from the catalogue.
        """
    )

    with gr.Row():

        with gr.Column(scale=1):

            image_input = gr.Image(
                type="filepath",
                label="Upload Saree Image"
            )

            message_input = gr.Textbox(
                label="What are you looking for?",
                placeholder="e.g. Find 5 sarees similar to this one",
                lines=3
            )

            search_button = gr.Button(
                "🔍 Find Similar Sarees",
                variant="primary"
            )

        with gr.Column(scale=1):

            output = gr.Markdown(
                label="TailorTalk Results"
            )

    search_button.click(
        fn=tailor_talk_search,
        inputs=[
            image_input,
            message_input
        ],
        outputs=output
    )


print("TailorTalk UI created successfully!")

demo.launch()

TailorTalk UI created successfully!
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
